# SARBLOH 001 — launcher

The agent is **not** in this notebook. It lives in the attached `sarbloh-src` dataset (the `sarbloh/` package,
built and uploaded by `scripts/kaggle_push.py`). This notebook only installs the game runtime, puts the package on
the path, sets the experiment's `CONFIG` and calls `sarbloh.harness.kaggle.main`.

- Save & Run: plays the offline public games and writes `results.json` / `summary.json` (not scored).
- Submission rerun: plays the hidden games through the Kaggle gateway (scored).

In [ ]:
import time
NOTEBOOK_START = time.time()

import json, os, subprocess, sys, sysconfig
from pathlib import Path

# The game runtime ships with the competition as offline wheels.
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-index", "--no-warn-conflicts",
     "--disable-pip-version-check", "--find-links",
     "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels", "arc-agi"],
    stdout=subprocess.DEVNULL,
)
os.environ["ONLY_RESET_LEVELS"] = "true"
os.environ["MPLBACKEND"] = "Agg"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    p for p in ["/usr/local/nvidia/lib64", *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if p
)

In [ ]:
# Find the sarbloh dataset by its marker file (Kaggle's mount path varies) and make the package importable
# here and in child processes.
marker = next(Path("/kaggle/input").rglob("sarbloh-bundle.json"), None)
assert marker is not None, "sarbloh-src dataset is not attached"
BUNDLE = marker.parent
sys.path.insert(0, str(BUNDLE))
(Path(sysconfig.get_paths()["purelib"]) / "sarbloh_bundle.pth").write_text(f"{BUNDLE}
")
print("sarbloh bundle:", BUNDLE, json.loads(marker.read_text()))

In [ ]:
# The experiment. Only overrides go here; defaults live in sarbloh/config.py (the Duck's settings).
EXPERIMENT = "E001_duck_port_fast_vllm"
CONFIG = {
    "label": EXPERIMENT,
    "vllm": {"profile": "fast", "fallback_profile": "duck"},
    # "games": ["ls20", "vc33"],            # offline subset for a quick check
    # "max_runtime_s_per_game": 1800.0,
}

In [ ]:
from sarbloh.harness.kaggle import main

summary = main(CONFIG, notebook_start_epoch=NOTEBOOK_START)
print("LEDGER_ROW " + json.dumps({"experiment": EXPERIMENT, **{k: v for k, v in summary.items() if k != "per_game"}}))